# MuJoCo Robotics Playground — Quick Start

This notebook walks you through:
1. Verifying the environment setup
2. Running a random agent
3. Training a PPO agent on CartPole
4. Plotting the training curve


In [ ]:
import sys
sys.path.insert(0, '..')  # run from notebooks/ dir

import numpy as np
import torch
print(f'PyTorch {torch.__version__}')

import mujoco
print(f'MuJoCo  {mujoco.__version__}')

## 1. Create an environment

In [ ]:
from envs import make_env

env = make_env('CartPole')
print('obs_space :', env.observation_space)
print('act_space :', env.action_space)

obs, info = env.reset(seed=0)
print('Initial obs:', obs)

## 2. Random agent baseline

In [ ]:
from agents import RandomAgent

agent = RandomAgent(env)
episode_returns = []

for ep in range(20):
    obs, _ = env.reset()
    done = False
    total = 0.0
    while not done:
        action = agent.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total += reward
    episode_returns.append(total)

print(f'Random agent — mean return: {np.mean(episode_returns):.1f} ± {np.std(episode_returns):.1f}')

## 3. Train PPO on CartPole (quick 50k steps)

In [ ]:
from agents.ppo import PPO, PPOConfig

env   = make_env('CartPole')
cfg   = PPOConfig(
    total_timesteps=50_000,
    n_steps=512,
    n_epochs=5,
    mini_batch_size=64,
    lr=3e-4,
    normalize_obs=True,
    log_dir='../logs',
)
agent = PPO(env, cfg)
agent.learn(log_interval=5)
print('Training complete.')

## 4. Plot training curve

In [ ]:
import matplotlib.pyplot as plt
import glob, pandas as pd

csv_files = sorted(glob.glob('../logs/CartPole_*/progress.csv'))
if csv_files:
    df = pd.read_csv(csv_files[-1])
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df['step'], df['ep_return_mean'], label='PPO')
    ax.set_xlabel('Environment steps')
    ax.set_ylabel('Episode return')
    ax.set_title('CartPole — PPO training curve')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No training log found — run the training cell first.')

## 5. Evaluate the trained agent

In [ ]:
eval_returns = []

for ep in range(10):
    obs, _ = env.reset()
    done = False
    total = 0.0
    while not done:
        action = agent.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total += reward
    eval_returns.append(total)

print(f'PPO agent (50k steps) — mean return: {np.mean(eval_returns):.1f} ± {np.std(eval_returns):.1f}')

## Next steps

- `02_sac_hopper.ipynb` — train SAC on the Hopper locomotion task
- `03_dreamer_quickstart.ipynb` — world model RL with DreamerV3
- `scripts/train.py` — command-line training with YAML configs
- `scripts/benchmark.py` — compare PPO vs SAC vs TD3
